# Exercises for Section 3.6 Statistical design and error analysis

This notebook contains the solutions to the exercises
from [Section 3.6 Statistical design and error analysis]()
in the **No Bullshit Guide to Statistics**.

### Notebook setup

In [1]:
# Ensure required Python modules are installed
%pip install --quiet numpy scipy seaborn pandas ministats

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Figures setup
plt.clf()  # needed otherwise `sns.set_theme` doesn't work
sns.set_theme(
    context="paper",
    style="whitegrid",
    palette="colorblind",
    rc={"figure.figsize": (6, 1.6)},
)
%config InlineBackend.figure_format = "retina"

<Figure size 640x480 with 0 Axes>

In [4]:
# Simplified int and float __repr__
np.set_printoptions(legacy='1.25')

In [5]:
# Download datasets/ directory if necessary
from ministats import ensure_datasets
ensure_datasets()

datasets/ directory already exists.


$\def\stderr#1{\mathbf{se}_{#1}}$
$\def\stderrhat#1{\hat{\mathbf{se}}_{#1}}$
$\newcommand{\Mean}{\textbf{Mean}}$
$\newcommand{\Var}{\textbf{Var}}$
$\newcommand{\Std}{\textbf{Std}}$
$\newcommand{\Freq}{\textbf{Freq}}$
$\newcommand{\RelFreq}{\textbf{RelFreq}}$
$\newcommand{\DMeans}{\textbf{DMeans}}$
$\newcommand{\Prop}{\textbf{Prop}}$
$\newcommand{\DProps}{\textbf{DProps}}$
$$
\newcommand{\CI}[1]{\textbf{CI}_{#1}}
\newcommand{\CIL}[1]{\textbf{L}_{#1}}
\newcommand{\CIU}[1]{\textbf{U}_{#1}}
\newcommand{\ci}[1]{\textbf{ci}_{#1}}
\newcommand{\cil}[1]{\textbf{l}_{#1}}
\newcommand{\ciu}[1]{\textbf{u}_{#1}}
$$
(this cell contains the macro definitions like $\stderr{\overline{\mathbf{x}}}$, $\stderrhat{}$, $\Mean$, ...)

## Exercises

### E3.44

<!-- {exercise:drug-discovery-test-design} -->

You're in charge of a clinical trial for a new drug.
The baseline model for the health indicator for untreated patients
is $I_0 \sim \mathcal{N}(\mu_{I_0}\!=\!100, \sigma_{I_0}\!=\!1)$.
You want to design a study to test if the new drug
increases the health indicator based on the hypotheses
$H_A: \mu > \mu_{I_0}$ and $H_0: \mu \leq \mu_{I_0}$.
The trial protocol requires $\alpha\!=\!0.05$ and $\beta\!=\!0.3$.
You consult with medical experts
and they say that the drug must improve the indicator by at least Cohen's $d=0.3$
to be clinically useful.
Find the minimum sample size for the clinical trial.

Hint: Use the `solve_power` function from the `TTestPower` class.


In [6]:
# @studentprompt
from statsmodels.stats.power import TTestPower
ttp = TTestPower()

# Known design parameters 
d = ...          # Effect size
alpha = ...      # False positive rate
power = ...      # Power

# Solve for the sample size given d, alpha, and power
... 

Ellipsis

In [7]:
# @solution
from statsmodels.stats.power import TTestPower
ttp = TTestPower()

# Known design parameters 
d = 0.3          # Effect size
alpha = 0.05     # False positive rate
power = 1 - 0.3  # Power

# Solve for the sample size given d, alpha, and power
n_trial = ttp.solve_power(effect_size=d, alpha=alpha,
                          power=power, alternative="larger")
n_trial

53.66191321182772

<!-- @solution -->
The minimum sample size for this clinical trial is $n=54$.

### E3.45

<!-- {exercise:left-tailed-test-kombucha} -->

You're developing a quality control test
to detect when the mean fill volume of kombucha bottles
is less than $\mu_{K_0} = 1000$ ml.
Assume overfull bottles are OK.
The standard deviation of kombucha volumes is $\sigma = 10$ ml.
Company policy states that false positive rate of 1 in 20 is acceptable.
The test must have at least 80% power
to detect batches of bottles where mean is less than $995$ ml.

**a)** Describe the two hypotheses $H_0$ and $H_A$.

**b)** What are the three design parameters given in the question?

**c)** Solve for the fourth parameter.

**d)** What is the cutoff value $\textrm{CV}_{\alpha}^-$ for the $t$-test?

**e)** Apply the test to a batch with mean $\overline{\textbf{k}}\!= 996$ ml
and standard deviation $s_{\textbf{k}}\!=\!11$ ml
by computing $t_{\textbf{k}}$ and compare it to $\textrm{CV}_{\alpha}^-$.


<!-- @solution -->
**a)**
The two competing hypotheses are

$$
 H_A: \mu\!<\!\mu_{K_0}
 \qquad \text{and} \qquad
 H_0: \mu\!\geq\!\mu_{K_0},
$$
 
where $\mu_{K_0}=1000$ is the expected mean when the production line is working as expected.

In [8]:
# @studentprompt
# b) Given design parameters
alpha = ...  # False positive rate
beta = ...   # False negative rate
Delta = ...  # Raw effect size 
d = ...      # Standardized effect size 

In [9]:
# @solution
# b) Given design parameters
alpha = 0.05  # 1 in 20
beta = 0.2    # 80% power

sigmaK = 10
Delta = 995 - 1000  # Raw effect size 
d = Delta / sigmaK  # Standardized effect size 
d

-0.5

In [10]:
# @studentprompt
# Solve for the sample size given d, alpha, and beta
from statsmodels.stats.power import TTestPower
ttp = TTestPower()
...

Ellipsis

In [11]:
# @solution
# c) Solve for approximate sample size using the z-test formula
from scipy.stats import norm
rvZ = norm(loc=0, scale=1)

z_u = rvZ.ppf(1-0.05)
z_l = rvZ.ppf(0.2)
n_approx = (z_u - z_l)**2 / d**2
n_approx

24.730228928079065

In [12]:
# @solution
# c) Solve for the sample size given d, alpha, and beta
from statsmodels.stats.power import TTestPower
ttp = TTestPower()
n_exact = ttp.solve_power(effect_size=d, alpha=alpha, power=1-beta,
                          alternative="smaller")
n_exact

26.137503817059283

In [13]:
# @studentprompt
# d) Calculate the cutoff value CV_alpha
# Round up sample size to nearest integer
n = ...

# Calculate CV_alpha from the t-distribution
from scipy.stats import t as tdist
rvT0 = ...
CV_alpha = ...

In [14]:
# @solution
# d) Calculate the cutoff value CV_alpha
# Round up to nearest integer
n = int(np.ceil(n_exact))
print("sample size =", n)

# Calculate CV_alpha from the t-distribution
from scipy.stats import t as tdist
rvT0 = tdist(df=n-1)
CV_alpha = rvT0.ppf(0.05)
CV_alpha

sample size = 27


-1.7056179197592733

In [15]:
# @studentprompt
# e) Apply the test to a batch with mean 996 and std 11
# Compute the estimated standard deviation
sehat = ...

# Compute the t-statistic
t = ...

In [16]:
# @solution
# e) Apply the test to a batch with mean 996 and std 11
# Compute the estimated standard deviation
sehat = 11 / np.sqrt(n)
print("sehat =", sehat)

# Compute the t-statistic
t = (996 - 1000) / sehat
t

sehat = 2.116950987028628


-1.8895099718933206

<!-- @solution -->
The $t$-statistic $-1.89$ is more extreme
than the cutoff value $\textrm{CV}_{\alpha}^- = -1.706$
so we reject $H_0$.
This batch seems to be underfull.

### E3.46

<!-- {exercise:sensitivity-of-one-sample-t-test-n-10} -->

Given an upper-tailed one-sample $t$-test
with cutoff parameter $\alpha\!=\!0.05$ and sample size $n\!=\!10$,
how large must the effect size be
so we can detect it with $80\%$ power?

Hint: Use the `solve_power` function from the `TTestPower` class.


In [17]:
# @studentprompt
from statsmodels.stats.power import TTestPower
ttp = TTestPower()

# Solve for the effect size given alpha, power, and sample size
...

Ellipsis

In [18]:
# @solution
from statsmodels.stats.power import TTestPower
ttp = TTestPower()

# Solve for the effect size given alpha, power, and sample size
ttp.solve_power(alpha=0.05, power=0.8, nobs=10, alternative="larger")

0.8528391375729821

<!-- @solution -->
An analysis based on samples of size $n=10$
will have 80\% power
if the true effect size is Cohen's $d=0.85$ or larger.

### E3.47

<!-- {exercise:power-of-two-sample-t-test-n-17} -->

Consider Bob's two-sample $t$-test for the electricity prices
with parameters $\alpha=0.05$
and sample sizes $n = m = 17$.
Calculate the power of the test assuming
the effect size is $d=1$.

Hint: Use the `power` method of the `TTestIndPower` class.

In [19]:
# @studentprompt
from statsmodels.stats.power import TTestIndPower
ttindp = TTestIndPower()

# Compute the power given alpha, n, and the effect size
...

Ellipsis

In [20]:
# @solution
from statsmodels.stats.power import TTestIndPower
ttindp = TTestIndPower()

# Compute the power given alpha, n, and the effect size
ttindp.power(alpha=0.05, nobs1=17, effect_size=1, alternative="two-sided")

0.8070367151472196

<!-- @solution -->
The power of the $\alpha=0.05$ test with sample sizes  $n=m=17$ is $81\%$.

### E3.48

<!-- {exercise:G-power-calc-two-sample-t-test} -->

Use G\*Power to calculate the power of Bob's two-sample $t$-test
with the same parameters as in the previous exercise,
$\alpha=0.05$, $n = m = 17$, and effect size $d=1$.


<!-- @solution -->
To calculate the power of Bob's two-sample $t$-test,
we use the option
"Means: Difference between two independent means (two groups)"
from the **Statistical test** drop-down,
and the option "Post hoc: Compute achieved power - given $\alpha$, sample size, and effect size"
from the **Type of power analysis** drop-down.

The right panel shows that the power of Bob's test is 81%,
which matches the result we obtained using `statsmodels`.